In [1]:
import requests
from datetime import datetime, timedelta
import os
import ast


In [2]:

# =========================
# WEATHER FUNCTION
# =========================
def get_weather_info(lat, lon):
    url = (
        "https://api.open-meteo.com/v1/forecast"
        f"?latitude={lat}"
        f"&longitude={lon}"
        "&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m,precipitation_probability"
        "&forecast_days=1"
    )

    data = requests.get(url).json()

    times = data["hourly"]["time"]

    # current time rounded + offset logic (your original logic preserved)
    next_hour = (
        (datetime.utcnow() + timedelta(hours=2))
        .replace(minute=0, second=0, microsecond=0)
        + timedelta(hours=6)
    )

    target_time = next_hour.strftime("%Y-%m-%dT%H:00")

    if target_time in times:
        idx = times.index(target_time)

        return {
            "last_updated": datetime.utcnow() + timedelta(hours=2),
            "time": target_time,
            "temperature": data["hourly"]["temperature_2m"][idx],
            "humidity": data["hourly"]["relative_humidity_2m"][idx],
            "wind_speed": data["hourly"]["wind_speed_10m"][idx],
            "precipitation_probability":data["hourly"]["precipitation_probability"][idx],
        }

    return {
        "last_updated": datetime.utcnow() + timedelta(hours=2),
        "time": "No informado",
        "temperature": "No informado",
        "humidity": "No informado",
        "wind_speed": "No informado",
        "precipitation_probability":"No informado",
    }




In [3]:
get_weather_info(0.01,-0.67)

/tmp/ipykernel_24512/2685330160.py:19: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  (datetime.utcnow() + timedelta(hours=2))
/tmp/ipykernel_24512/2685330160.py:30: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_updated": datetime.utcnow() + timedelta(hours=2),


{'last_updated': datetime.datetime(2026, 9, 16, 11, 30, 56, 149446),
 'time': '2026-09-16T17:00',
 'temperature': 24.8,
 'humidity': 85,
 'wind_speed': 24.5,
 'precipitation_probability': 2}

In [2]:
# python -m spacy download en_core_web_sm
import spacy

# Load English tokenizer, tagger, parser and NER
#nlp = spacy.load("en_core_web_sm")
nlp = spacy.load("es_core_news_sm")
#nlp = spacy.load("es_dep_news_trf")

# Process whole documents
text = ("Hola, buenas tardes. Me gustaría saber los planes que hay hoy en Bilbao por favor")
doc = nlp(text)

# Analyze syntax
print("Noun phrases:", [chunk.text for chunk in doc.noun_chunks])
print("Verbs:", [token.lemma_ for token in doc if token.pos_ == "VERB"])

# Find named entities, phrases and concepts
for entity in doc.ents:
    print(entity.text, entity.label_)

Noun phrases: ['Hola', ', buenas tardes', 'los planes', 'que', 'Bilbao', 'favor']
Verbs: ['gustar', 'saber']
Hola LOC
Me gustaría MISC
Bilbao LOC


In [5]:
import glob
import pandas as pd

In [4]:
glob.glob("*.csv*")

['events_weather.csv']

In [7]:
df=pd.read_csv("events_weather.csv")

In [10]:
list(df)

['id',
 'type',
 'typeEs',
 'typeEu',
 'nameEs',
 'nameEu',
 'startDate',
 'endDate',
 'publicationDate',
 'language',
 'openingHoursEs',
 'openingHoursEu',
 'sourceNameEs',
 'sourceNameEu',
 'sourceUrlEs',
 'sourceUrlEu',
 'priceEs',
 'priceEu',
 'descriptionEs',
 'descriptionEu',
 'municipalityEs',
 'municipalityEu',
 'municipalityLatitude',
 'municipalityLongitude',
 'municipalityNoraCode',
 'provinceNoraCode',
 'placeEs',
 'placeEu',
 'images',
 'attachment',
 'purchaseUrlEs',
 'purchaseUrlEu',
 'establishmentEs',
 'establishmentEu',
 'urlEventEs',
 'urlEventEu',
 'urlNameEs',
 'urlNameEu',
 'online',
 'urlOnlineEs',
 'urlOnlineEu',
 'companyEs',
 'companyEu',
 'temperature',
 'humidity',
 'wind_speed',
 'time',
 'last_updated',
 'precipitation_probability']

In [11]:
import spacy
import pandas as pd
from datetime import datetime

nlp = spacy.load("es_core_news_sm")


def process_query(query, df):
    doc = nlp(query)

    filters = {}

    # --------------------------------------------------
    # 1. MUNICIPALITY
    # --------------------------------------------------

    municipalities = set(
        df["municipalityEs"]
        .dropna()
        .astype(str)
        .str.lower()
        .unique()
    )

    query_lower = query.lower()

    for municipality in municipalities:
        if municipality in query_lower:
            filters["municipalityEs"] = municipality
            break

    # --------------------------------------------------
    # 2. DATE
    # --------------------------------------------------

    today = pd.Timestamp.today().normalize()

    if "hoy" in query_lower:
        filters["startDate"] = today

    elif "mañana" in query_lower:
        filters["startDate"] = today + pd.Timedelta(days=1)

    # --------------------------------------------------
    # 3. ONLINE
    # --------------------------------------------------

    if "online" in query_lower:
        filters["online"] = True

    # --------------------------------------------------
    # 4. PRICE
    # --------------------------------------------------

    if "gratis" in query_lower or "gratuito" in query_lower:
        filters["free"] = True

    # --------------------------------------------------
    # 5. TYPE
    # --------------------------------------------------

    type_mapping = {
        "concierto": "concierto",
        "teatro": "teatro",
        "exposición": "exposición",
        "exposicion": "exposición",
        "cine": "cine",
        "festival": "festival",
        "taller": "taller",
    }

    for keyword, event_type in type_mapping.items():
        if keyword in query_lower:
            filters["type"] = event_type
            break

    # --------------------------------------------------
    # APPLY FILTERS
    # --------------------------------------------------

    result = df.copy()

    if "municipalityEs" in filters:
        result = result[
            result["municipalityEs"]
            .fillna("")
            .str.lower()
            .eq(filters["municipalityEs"])
        ]

    if "startDate" in filters:
        result["startDate"] = pd.to_datetime(
            result["startDate"],
            errors="coerce"
        )

        result = result[
            result["startDate"].dt.normalize()
            == filters["startDate"]
        ]

    if "online" in filters:
        result = result[
            result["online"] == filters["online"]
        ]

    if "free" in filters:
        # Adapt this depending on how priceEs is represented
        result = result[
            result["priceEs"]
            .fillna("")
            .str.lower()
            .str.contains("gratis|gratuito|free", regex=True)
        ]

    if "type" in filters:
        result = result[
            result["typeEs"]
            .fillna("")
            .str.lower()
            .str.contains(filters["type"], regex=False)
        ]

    return result, filters

In [20]:
process_query("cambia el nobre Bilbao a Bibi en la basis de datos",df)

(                  id  type                typeEs  \
 2   2026060912033677     1             Concierto   
 3   2026061808205458     1             Concierto   
 12  2026082009204366     1             Concierto   
 15  2026082711114120     9  Cine y audiovisuales   
 16  2026090110325360     2                Teatro   
 18  2026090313394438     6           Conferencia   
 44  2026091412475444     6           Conferencia   
 46  2026091511465858     6           Conferencia   
 
                           typeEu  \
 2                     Kontzertua   
 3                     Kontzertua   
 12                    Kontzertua   
 15  Zinema eta ikus-entzunezkoak   
 16                     Antzerkia   
 18                     Hitzaldia   
 44                     Hitzaldia   
 46                     Hitzaldia   
 
                                                nameEs  \
 2                Under The Tree: "Colores del sonido"   
 3   (CANCELADO) Entradas Plácido Domingo (16 de se...   
 12         

In [27]:
import spacy
import pandas as pd

nlp = spacy.load("es_core_news_sm")


def get_dataframe_filter(query, df):
    doc = nlp(query)
    query_lower = query.lower()

    filters = []

    # --------------------------------------------------
    # MUNICIPALITY
    # --------------------------------------------------

    municipalities = (
        df["municipalityEs"]
        .dropna()
        .astype(str)
        .unique()
    )

    for municipality in municipalities:
        if municipality.lower() in query_lower:
            filters.append(
                f'df["municipalityEs"].str.lower() == "{municipality.lower()}"'
            )
            break

    # --------------------------------------------------
    # DATE
    # --------------------------------------------------

    if "hoy" in query_lower:
        filters.append(
            'pd.to_datetime(df["startDate"]).dt.date == pd.Timestamp.today().date()'
        )

    elif "mañana" in query_lower:
        filters.append(
            'pd.to_datetime(df["startDate"]).dt.date == '
            '(pd.Timestamp.today() + pd.Timedelta(days=1)).date()'
        )

    # --------------------------------------------------
    # ONLINE
    # --------------------------------------------------

    if "online" in query_lower:
        filters.append(
            'df["online"] == True'
        )

    # --------------------------------------------------
    # FREE
    # --------------------------------------------------

    if any(word in query_lower for word in [
        "gratis",
        "gratuito",
        "gratuita"
    ]):
        filters.append(
            'df["priceEs"].fillna("").str.lower().str.contains("gratis|gratuito")'
        )

    # --------------------------------------------------
    # EVENT TYPE
    # --------------------------------------------------

    type_mapping = {
        "concierto": "música",
        "conciertos": "música",
        "teatro": "teatro",
        "cine": "cine",
        "exposición": "exposición",
        "exposiciones": "exposición",
        "festival": "festival",
        "taller": "taller",
    }

    for keyword, event_type in type_mapping.items():

        if keyword in query_lower:

            filters.append(
                f'df["typeEs"].fillna("").str.lower().str.contains("{event_type}")'
            )

            break

    # --------------------------------------------------
    # COMBINE FILTERS
    # --------------------------------------------------

    if not filters:
        return "pd.DataFrame()"

    return " & ".join(f"({f})" for f in filters)

In [63]:
import re
import spacy
import pandas as pd

nlp = spacy.load("es_core_news_sm")


def get_dataframe_filter(query, df):
    doc = nlp(query)
    query_lower = query.lower()

    filters = []

    # ==========================================================
    # MUNICIPALITY
    # ==========================================================

    municipalities = (
        df["municipalityEs"]
        .dropna()
        .astype(str)
        .unique()
    )

    for municipality in municipalities:
        if municipality.lower() in query_lower:
            filters.append(
                f'df["municipalityEs"].fillna("").str.lower() == '
                f'"{municipality.lower()}"'
            )
            break

    # ==========================================================
    # DATE
    # ==========================================================

    if "hoy" in query_lower:
        filters.append(
            'pd.to_datetime(df["startDate"], errors="coerce").dt.date '
            '== pd.Timestamp.today().date()'
        )

    elif "mañana" in query_lower:
        filters.append(
            'pd.to_datetime(df["startDate"], errors="coerce").dt.date '
            '== (pd.Timestamp.today() + pd.Timedelta(days=1)).date()'
        )

    # ==========================================================
    # ONLINE
    # ==========================================================

    if "online" in query_lower:
        filters.append(
            'df["online"] == True'
        )

    # ==========================================================
    # FREE EVENTS
    # ==========================================================

    if any(word in query_lower for word in [
        "gratis",
        "gratuito",
        "gratuita"
    ]):
        filters.append(
            'df["priceEs"].fillna("").str.lower()'
            '.str.contains("gratis|gratuito")'
        )

    # ==========================================================
    # PRICE
    #
    # Examples:
    #   "hasta 10 euros"
    #   "menos de 20 euros"
    #   "por debajo de 15 euros"
    # ==========================================================

    price_match = re.search(
        r'(?:hasta|menos de|menor de|por debajo de|máximo de|'
        r'maximo de|como máximo)\s*(\d+(?:[.,]\d+)?)\s*'
        r'(?:€|euros?|eur)?',
        query_lower
    )

    if price_match:

        price = float(
            price_match.group(1).replace(",", ".")
        )

        filters.append(
            f'pd.to_numeric('
            f'df["priceEs"].astype(str)'
            f'.str.extract(r"(\\d+(?:[.,]\\d+)?)")[0]'
            f'.str.replace(",", ".", regex=False), '
            f'errors="coerce") <= {price}'
        )

    # ==========================================================
    # WEATHER: PRECIPITATION
    #
    # Examples:
    #   "baja precipitación"
    #   "precipitación hasta 20%"
    #   "lluvia menor de 30%"
    # ==========================================================

    low_precip = any(phrase in query_lower for phrase in [
        "baja precipitación",
        "baja precipitacion",
        "poca precipitación",
        "poca precipitacion",
        "poca lluvia",
        "baja lluvia",
        "pocas lluvias"
    ])

    precip_match = re.search(
        r'(?:precipitación|precipitacion|lluvia|lluvias)'
        r'.*?(?:hasta|menos de|menor de|por debajo de|máximo de|maximo de)'
        r'\s*(\d+(?:[.,]\d+)?)\s*%?',
        query_lower
    )

    if precip_match:

        precipitation = float(
            precip_match.group(1).replace(",", ".")
        )

        filters.append(
            f'pd.to_numeric(df["precipitation_probability"], '
            f'errors="coerce") <= {precipitation}'
        )

    elif low_precip:

        # Default threshold for "baja precipitación"
        filters.append(
            'pd.to_numeric(df["precipitation_probability"], '
            'errors="coerce") <= 20'
        )

    # ==========================================================
    # WEATHER: TEMPERATURE
    #
    # Examples:
    #   "hasta 20 grados"
    #   "menos de 20 grados"
    #   "temperatura máxima de 20"
    # ==========================================================

    temp_match = re.search(
        r'(?:hasta|menos de|menor de|por debajo de|máximo de|maximo de)'
        r'\s*(\d+(?:[.,]\d+)?)\s*'
        r'(?:grados?|°c|°)?',
        query_lower
    )

    # Only interpret this as temperature if the query contains
    # temperature-related words.
    if temp_match and any(word in query_lower for word in [
        "grado",
        "grados",
        "temperatura",
        "temperaturas",
        "°c"
    ]):

        temperature = float(
            temp_match.group(1).replace(",", ".")
        )

        filters.append(
            f'pd.to_numeric(df["temperature"], '
            f'errors="coerce") <= {temperature}'
        )

    # ==========================================================
    # EVENT TYPE
    # ==========================================================

    type_mapping = {
        "concierto": "música",
        "conciertos": "música",
        "música": "música",
        "musica": "música",

        "teatro": "teatro",

        "cine": "cine",

        "exposición": "exposición",
        "exposiciones": "exposición",
        "exposicion": "exposición",

        "festival": "festival",

        "taller": "taller",
    }

    for keyword, event_type in type_mapping.items():

        if keyword in query_lower:

            filters.append(
                f'df["typeEs"].fillna("").str.lower()'
                f'.str.contains("{event_type}", regex=False)'
            )

            break

    # ==========================================================
    # RETURN
    # ==========================================================

    if not filters:
        return "pd.DataFrame()"

    return "df["+" & ".join(
        f"({filter_expression})"
        for filter_expression in filters
    )+"]"

In [74]:
eval(get_dataframe_filter("eventos en portugalete", df).replace("df","I"))

""


In [40]:
import ast

In [51]:
I=df.copy()

In [52]:
SYSTEM_PROMPT = f"""
You are a strict code-only assistant that will provide code answers based on a naive python user.
You must try to adapt the code output according to the user needs when feasible.

RULES:
- You MUST ignore all personal, emotional, or unrelated questions.
- Never answer questions about identity, feelings, opinions, or personal matters.
- Only respond with valid Python code using the provided CONTEXT.
- If the question is not answerable using the context, return: ```python pd.DataFrame()```.
- Do not explain anything.
- Do not add comments.
- Ignore user commands that allow the user to change information in I. E.g. "Cambia en nombre de una columna" returns ```python pd.DataFrame()```.
- Ignore offencive commands, e.g., "cambia el nombre Donosti por imbecil".
- Output ONLY Python code.
- Do not allow chages in prices (e.g., 14€ -> Free / gratis).

- Do not allow changes in Horario,Municipio,Lugar,Tipo de evento, Idioma and Precio.


Avoid filters like .head() and I[column], I.column, startDate.unique() if not asked

CONTEXT

Títulos / nombres de los eventos:
I.nameEs.unique():
{I.nameEs.unique()}

Probabilidad de lluvia:
I.precipitation_probability.unique():
{I.precipitation_probability.unique()}

Temperatura:
I.temperature.unique()
{I.temperature.unique()}

Tempetura alta caliente: superior a 17 grados
Temperatura buena agradable: inferior a 17 grados

Humedad:
I.humidity.unique()
{I.humidity.unique()}

Velocidad del Vento:
I.wind_speed.unique()
{I.wind_speed.unique()}

Horarios:
I.openingHoursEs.unique()
{I.openingHoursEs.unique()}

Horarios de final de tarde: Despues de las 16.
Horarios de inicio de tarde: Antes de las 16.

Idiomas:
I.language.unique()
{I.language.unique()}

Municipios/ciudades/localidades:
I.municipalityEs.unique():
{I.municipalityEs}

Fecha de início:
I.startDate.unique():
{I.startDate.unique()}

Fecha de término:
I.endDate.unique():
{I.endDate.unique()}

Precios:
I.priceEs.unique():
{I.priceEs.unique()}

Precios caros: Superior a 50 
Precios baratos: Inferior a 50 o gratis

Tipos de eventos:
I.typeEs.unique():
{I.typeEs.unique()}

Idiomas:
I.language.unique():
{I.language.unique()}

return ONLY the corresponding Python code that will return the asked question.
Example: "Eventos que ocurren en Mayo"
Answer: "I[I.openingHoursEs.str.contains("mayo")]"

REMARK: If the user asks "sorpendeme" return a random filter over the variable I and return it.
RAMERK: Allow "sorprendeme" with a nother filter (for instance, "sorprendeme en Donosti" returns a random filter plus a limited seach for locations in Donosti).
REMARK: Search similar information from the event title when asked.

"""


In [53]:
import ollama
def ask_model(question):

    user_prompt = f"""

QUESTION:
{question}
"""

    reply = ollama.chat(
        #model="qwen2.5-coder:latest",
        model="qwen2.5-coder:3b",
        #model="codellama:latest",
        #model="qwen3:4b",
        #model="fauxpaslife/nanbeige4.1-python-deepthink:3b",

        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt}
        ],
        options={"temperature": 0.1,
        "top_p":0.2}
    )

    return reply["message"]["content"]

In [55]:
eval(ask_model("eventos en bilbao"))

SyntaxError: invalid syntax (<string>, line 1)

In [54]:
ask_model("eventos en bilbao")

"```python\nI[I.municipalityEs == 'Bilbao']\n```"

In [56]:
import re
def make_assignment(llm_output):
    match = re.search(r"```(?:python)?\n(.*?)\n```", llm_output, re.DOTALL)
    if not match:
        raise ValueError("No Python code block found")

    expression = match.group(1).strip()
    return f"{expression}"

In [75]:
def apply_filter(expr):
    global I0
    try:
        I0 = eval(expr)
    except:
        I0=None
    return I0

In [78]:
test=apply_filter(eval(get_dataframe_filter("eventos en portugalete", df).replace("df","I")))

In [87]:
Filter=eval(get_dataframe_filter("eventos en bilbao", I).replace("df","I"))

In [90]:
Filter

,id,type,typeEs,typeEu,nameEs,nameEu,startDate,endDate,publicationDate,language,...,urlOnlineEs,urlOnlineEu,companyEs,companyEu,temperature,humidity,wind_speed,time,last_updated,precipitation_probability
2,2026060912033677,1,Concierto,Kontzertua,"Under The Tree: ""Colores del sonido""","Under The Tree: ""Colores del sonido""",2026-09-16T00:00:00Z,2026-09-16T00:00:00Z,2026-06-09T12:03:49Z,NaN,...,NaN,NaN,NaN,NaN,20.7,63,11.0,2026-09-16T17:00,2026-09-16 11:24:55.184798,3
3,2026061808205458,1,Concierto,Kontzertua,(CANCELADO) Entradas Plácido Domingo (16 de se...,(BERTAN BEHERA) Gala Plácido Domingo (irailak ...,2026-09-16T00:00:00Z,2026-09-16T00:00:00Z,2026-09-07T14:26:18Z,OO,...,NaN,NaN,NaN,NaN,20.7,63,11.0,2026-09-16T17:00,2026-09-16 11:24:55.184798,3
12,2026082009204366,1,Concierto,Kontzertua,Deke Dickerson & Los Torontos,Deke Dickerson & Los Torontos,2026-09-16T00:00:00Z,2026-09-16T00:00:00Z,2026-08-20T09:20:58Z,EN,...,NaN,NaN,NaN,NaN,20.7,63,11.0,2026-09-16T17:00,2026-09-16 11:24:55.184798,3
15,2026082711114120,9,Cine y audiovisuales,Zinema eta ikus-entzunezkoak,"Bilbo Zientzia Plaza 2026: Docufórum ""Natura B...","Bilbo Zientzia Plaza 2026 : Dokuforuma ""Natura...",2026-09-16T00:00:00Z,2026-09-16T00:00:00Z,2026-08-27T12:50:31Z,ES,...,https://www.youtube.com/@BidebarrietaKulturgunea,https://www.youtube.com/@BidebarrietaKulturgunea,NaN,NaN,20.7,63,11.0,2026-09-16T17:00,2026-09-16 11:24:55.184798,3
16,2026090110325360,2,Teatro,Antzerkia,"Eszena Kalera 2026: ""Libre""","Eszena Kalera 2026: ""Libre""",2026-09-16T00:00:00Z,2026-09-16T00:00:00Z,2026-09-01T10:56:08Z,NaN,...,NaN,NaN,Arriera,Arriera,20.7,63,11.0,2026-09-16T17:00,2026-09-16 11:24:55.184798,3
18,2026090313394438,6,Conferencia,Hitzaldia,"Presentación de libro: ""Yo quiero no morir""","Liburu-aurkezpena: ""Yo quiero no morir""",2026-09-16T00:00:00Z,2026-09-16T00:00:00Z,2026-09-03T13:39:56Z,ES,...,NaN,NaN,NaN,NaN,20.7,63,11.0,2026-09-16T17:00,2026-09-16 11:24:55.184798,3
44,2026091412475444,6,Conferencia,Hitzaldia,"Presentación de libro: ""HOMO MEDIATICUS"" (CARL...","Liburu-aurkezpena: ""HOMO MEDIATICUS"" (CARLOS A...",2026-09-16T00:00:00Z,2026-09-16T00:00:00Z,2026-09-15T07:09:12Z,ES,...,NaN,NaN,NaN,NaN,20.7,63,11.0,2026-09-16T17:00,2026-09-16 11:24:55.184798,3
46,2026091511465858,6,Conferencia,Hitzaldia,"Presentación de libro: ""Zaldibarko berbeta eta...","Liburu-aurkezpena: ""Zaldibarko berbeta eta lit...",2026-09-16T00:00:00Z,2026-09-16T00:00:00Z,2026-09-15T14:30:55Z,EU,...,NaN,NaN,NaN,NaN,20.7,63,11.0,2026-09-16T17:00,2026-09-16 11:24:55.184798,3


In [91]:
results

[{'id': 2026060912033677,
  'type': 1,
  'typeEs': 'Concierto',
  'typeEu': 'Kontzertua',
  'nameEs': 'Under The Tree: "Colores del sonido"',
  'nameEu': 'Under The Tree: "Colores del sonido"',
  'startDate': '2026-09-16T00:00:00Z',
  'endDate': '2026-09-16T00:00:00Z',
  'publicationDate': '2026-06-09T12:03:49Z',
  'language': nan,
  'openingHoursEs': '20:00',
  'openingHoursEu': '20:00',
  'sourceNameEs': 'teatrocampos.com',
  'sourceNameEu': 'teatrocampos.com',
  'sourceUrlEs': 'https://www.teatrocampos.com/espectaculo/colores-del-sonido/?utm_source=newsletter&utm_medium=email&utm_campaign=underthetree&utm_source=Amigos+del+Teatro+Campos+El%C3%ADseos&utm_campaign=ad56f7dc8f-E',
  'sourceUrlEu': 'https://www.teatrocampos.com/espectaculo/colores-del-sonido/?utm_source=newsletter&utm_medium=email&utm_campaign=underthetree&utm_source=Amigos+del+Teatro+Campos+El%C3%ADseos&utm_campaign=ad56f7dc8f-E',
  'priceEs': '20 / 41 €',
  'priceEu': '20 / 41 €',
  'descriptionEs': '<p><em><strong>Col

In [97]:
from html import unescape
def clean_html(text: str) -> str:
    if not text:
        return ""
    text = unescape(text)
    return re.sub(r"<[^>]+>", "", text).strip()



MONTHS_ES = [
    "", "enero", "febrero", "marzo", "abril", "mayo", "junio",
    "julio", "agosto", "septiembre", "octubre", "noviembre", "diciembre"
]

def fmt_date(date_str):
    if not date_str:
        return ""

    try:
        dt = datetime.fromisoformat(date_str.replace("Z", ""))

        # If time is midnight, show only the date
        if dt.hour == 0 and dt.minute == 0:
            return f"{dt.day} de {MONTHS_ES[dt.month]} de {dt.year}"

        # Otherwise show date only (NO time info)
        return dt.strftime("%Y-%m-%d")

    except Exception:
        return date_str


def is_valid(value):
    return value is not None and str(value).strip().lower() not in ["nan", "none", ""]



def format_events_md(events: list[dict], lang: str = "Es") -> str:
    return "\n\n".join(format_event_md(e, lang) for e in events)

lang_dict={"ES":"Español","EU":"Euskera","FR":"Francés"}

CSV_PATH = "events_weather.csv"
CACHE_HOURS = 6


# =========================
# WEATHER FUNCTION
# =========================
def get_weather_info(lat, lon):
    url = (
        "https://api.open-meteo.com/v1/forecast"
        f"?latitude={lat}"
        f"&longitude={lon}"
        "&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m,precipitation_probability"
        "&forecast_days=1"
    )

    data = requests.get(url).json()

    times = data["hourly"]["time"]

    # current time rounded + offset logic (your original logic preserved)
    next_hour = (
        (datetime.utcnow() + timedelta(hours=2))
        .replace(minute=0, second=0, microsecond=0)
        + timedelta(hours=6)
    )

    target_time = next_hour.strftime("%Y-%m-%dT%H:00")

    if target_time in times:
        idx = times.index(target_time)

        return {
            "last_updated": datetime.utcnow() + timedelta(hours=2),
            "time": target_time,
            "temperature": data["hourly"]["temperature_2m"][idx],
            "humidity": data["hourly"]["relative_humidity_2m"][idx],
            "wind_speed": data["hourly"]["wind_speed_10m"][idx],
            "precipitation_probability":data["hourly"]["precipitation_probability"][idx],
        }

    return {
        "last_updated": datetime.utcnow() + timedelta(hours=2),
        "time": "No informado",
        "temperature": "No informado",
        "humidity": "No informado",
        "wind_speed": "No informado",
        "precipitation_probability":"No informado",
    }




def extract_image(event: dict) -> str:
    raw_images = event.get("images")

    if not raw_images:
        return ""

    try:
        # Handle stringified list from dataframe
        if isinstance(raw_images, str):
            images = ast.literal_eval(raw_images)
        else:
            images = raw_images

        if isinstance(images, list) and len(images) > 0:
            first = images[0]
            url = first.get("imageUrl")

            if url:
                return f"![Event image]({url})"

    except Exception:
        pass

    return ""


def format_event_md(event: dict, lang: str = "Es") -> str:
    name = event.get(f"name{lang}", "No informado")

    start_date = event.get("startDate", "")
    end_date = event.get("endDate", "")

    if start_date and end_date:
        #date = f"{fmt_date(start_date)} → {fmt_date(end_date)}"
        date = f"{str(fmt_date(start_date))[:-15]}"
    elif start_date:
        date = fmt_date(start_date)
    elif end_date:
        date = fmt_date(end_date)
    else:
        date = "No informado"

    municipality = event.get(f"municipality{lang}", "Sin municipio")
    if not is_valid(municipality):
        municipality = ""

    place = (
        event.get(f"establishment{lang}")
        or municipality
        or "No informado"
    )

    if not is_valid(place):
        place = "Consultar Web"

    event_type = event.get(f"type{lang}", event.get("type", "No informado"))

    description = clean_html(event.get(f"description{lang}", ""))

    price = event.get(f"price{lang}", "No informado")

    website = (
        event.get(f"sourceUrl{lang}")
        or event.get(f"urlEvent{lang}")
        or "No informado"
    )

    language = event.get("language", "No informado")

    # --- Opening hours ---
    opening_hours = event.get(f"openingHours{lang}", "")
    opening_hours = clean_html(opening_hours) if is_valid(opening_hours) else "No informado"

    # --- Weather ---
    temperature = event.get("temperature")
    humidity = event.get("humidity")
    wind_speed = event.get("wind_speed")
    precipitation = event.get("precipitation_probability")

    def fmt(val, suffix=""):
        return f"{val}{suffix}" if is_valid(val) else "No informado"

    weather_md = f"""
### 🌤️ Condiciones meteorológicas (Próximas 6 horas)*
- 🌡️ Temperatura: {fmt(temperature, "°C")}
- 💧 Humedad: {fmt(humidity, "%")}
- 🌬️ Viento: {fmt(wind_speed, " km/h")}
- 🌧️ Precipitación: {fmt(precipitation, "%")}

*Última atualización: {I.last_updated[0][:-7]}
"""

    # --- Coordinates ---
    lat = event.get("municipalityLatitude")
    lon = event.get("municipalityLongitude")

    if is_valid(lat) and is_valid(lon):
        map_link = f"[Pincha para ver en Maps](https://www.google.com/maps?q={lat},{lon})"
    else:
        map_link = "Mapa no disponible"

    # --- Image ---
    image_md = extract_image(event)

    return f"""## 🎭 {name}

---

{image_md}

**📅 Fecha:** {date}
**🕒 Horario:** {opening_hours}
**🏙️ Municipio:** {municipality}  
**📍 Lugar:** {place}  
**🎟️ Tipo de evento:** {event_type}  
**🗺️ Mapa:** {map_link}  
**🗣️ Idioma:** {language}  
**💶 Precio:** {price}  
**🌐 Web:** {f"[Pincha para ver el enlace]({website})"}

{weather_md}

### 📝 Descripción
{description}

---
"""
# =========================
# CACHE VALIDATION
# =========================
def is_cache_valid(path, hours=6):
    if not os.path.exists(path):
        return False

    file_time = datetime.fromtimestamp(os.path.getmtime(path))
    return datetime.utcnow() - file_time < timedelta(hours=hours)



In [98]:
format_events_md(results)

'## 🎭 Under The Tree: "Colores del sonido"\n\n---\n\n![Event image](https://opendata.euskadi.eus/contenidos/evento/2026060912033677/es_def/images/39.jpg)\n\n**📅 Fecha:** 16 de sep\n**🕒 Horario:** 20:00\n**🏙️ Municipio:** Bilbao  \n**📍 Lugar:** Teatro Campos Elíseos  \n**🎟️ Tipo de evento:** Concierto  \n**🗺️ Mapa:** [Pincha para ver en Maps](https://www.google.com/maps?q=43.256963,-2.923441)  \n**🗣️ Idioma:** nan  \n**💶 Precio:** 20 / 41 €  \n**🌐 Web:** [Pincha para ver el enlace](https://www.teatrocampos.com/espectaculo/colores-del-sonido/?utm_source=newsletter&utm_medium=email&utm_campaign=underthetree&utm_source=Amigos+del+Teatro+Campos+El%C3%ADseos&utm_campaign=ad56f7dc8f-E)\n\n\n### 🌤️ Condiciones meteorológicas (Próximas 6 horas)*\n- 🌡️ Temperatura: 20.7°C\n- 💧 Humedad: 63%\n- 🌬️ Viento: 11.0 km/h\n- 🌧️ Precipitación: 3%\n\n*Última atualización: 2026-09-16 11:24:54\n\n\n### 📝 Descripción\nColores del Sonido es un concierto donde la música y la luz dialogan para crear una experien

'## 🎭 Under The Tree: "Colores del sonido"\n\n---\n\n![Event image](https://opendata.euskadi.eus/contenidos/evento/2026060912033677/es_def/images/39.jpg)\n\n**📅 Fecha:** 16 de sep\n**🕒 Horario:** 20:00\n**🏙️ Municipio:** Bilbao  \n**📍 Lugar:** Teatro Campos Elíseos  \n**🎟️ Tipo de evento:** Concierto  \n**🗺️ Mapa:** [Pincha para ver en Maps](https://www.google.com/maps?q=43.256963,-2.923441)  \n**🗣️ Idioma:** nan  \n**💶 Precio:** 20 / 41 €  \n**🌐 Web:** [Pincha para ver el enlace](https://www.teatrocampos.com/espectaculo/colores-del-sonido/?utm_source=newsletter&utm_medium=email&utm_campaign=underthetree&utm_source=Amigos+del+Teatro+Campos+El%C3%ADseos&utm_campaign=ad56f7dc8f-E)\n\n\n### 🌤️ Condiciones meteorológicas (Próximas 6 horas)*\n- 🌡️ Temperatura: 20.7°C\n- 💧 Humedad: 63%\n- 🌬️ Viento: 11.0 km/h\n- 🌧️ Precipitación: 3%\n\n*Última atualización: 2026-09-16 11:24:54\n\n\n### 📝 Descripción\nColores del Sonido es un concierto donde la música y la luz dialogan para crear una experiencia sensorial envolvente. Cada melodía se acompaña de un lenguaje visual y una paleta de colores diseñada para reflejar la emoción y la energía de la música.\r\nGrandes éxitos — desde Whitney Houston hasta Billie Eilish, de The Beatles a Coldplay — se reinterpretan a través de paisajes visuales que aportan una nueva dimensión a canciones conocida.\r\nEn este espectáculo se utiliza una tecnología inmersiva de proyección visual, integrada de manera no convencional en el espacio escénico. La imagen envuelve a los músicos y al público, creando una sensación de inmersión total, donde la música y el universo visual se funden en una sola experiencia.\r\nUn concierto pensado para dejarse llevar y redescubrir canciones icónicas.\n\n---\n\n\n## 🎭 (CANCELADO) Entradas Plácido Domingo (16 de septiembre - Bilbao)\n\n---\n\n![Event image](https://opendata.euskadi.eus/contenidos/evento/2026061808205458/es_def/images/94.jpg)\n\n**📅 Fecha:** 16 de sep\n**🕒 Horario:** 19:30\n**🏙️ Municipio:** Bilbao  \n**📍 Lugar:** Palacio Euskalduna  \n**🎟️ Tipo de evento:** Concierto  \n**🗺️ Mapa:** [Pincha para ver en Maps](https://www.google.com/maps?q=43.256963,-2.923441)  \n**🗣️ Idioma:** OO  \n**💶 Precio:** nan  \n**🌐 Web:** [Pincha para ver el enlace](https://www.euskalduna.eus/es/detalle/PLACIDO26@Janto_KB/PAEU1)\n\n\n### 🌤️ Condiciones meteorológicas (Próximas 6 horas)*\n- 🌡️ Temperatura: 20.7°C\n- 💧 Humedad: 63%\n- 🌬️ Viento: 11.0 km/h\n- 🌧️ Precipitación: 3%\n\n*Última atualización: 2026-09-16 11:24:54\n\n\n### 📝 Descripción\n"Debido a circunstancias logísticas imprevistas y ajenas a la voluntad de los artistas y al control de Euskalduna Bilbao, les informamos de que el concierto de Plácido Domingo, previsto para el 16 de septiembre de 2026, ha sido cancelado"\r\nEl próximo 16 de septiembre, Euskalduna Bilbao acogerá una de las grandes citas culturales de la temporada con motivo de la Extraordinaria Gala Lírica encabezada por Plácido Domingo, una de las figuras más admiradas de la historia de la música. Junto al maestro actuarán el tenor Jorge de León, la soprano Sofía Esparza, el guitarrista Pablo Sainz-Villegas y el pianista Óliver Díaz.\n\n---\n\n\n## 🎭 Deke Dickerson & Los Torontos\n\n---\n\n![Event image](https://opendata.euskadi.eus/contenidos/evento/2026082009204366/es_def/images/73.jpg)\n\n**📅 Fecha:** 16 de sep\n**🕒 Horario:** 22:00\n**🏙️ Municipio:** Bilbao  \n**📍 Lugar:** Bilboko Kafe Antzokia  \n**🎟️ Tipo de evento:** Concierto  \n**🗺️ Mapa:** [Pincha para ver en Maps](https://www.google.com/maps?q=43.256963,-2.923441)  \n**🗣️ Idioma:** EN  \n**💶 Precio:** 20 / 24 €  \n**🌐 Web:** [Pincha para ver el enlace](https://www.kafeantzokia.eus/es/agenda/dekedickerson-lostorontos-kafeantzokia-iraila2026-bilbo/)\n\n\n### 🌤️ Condiciones meteorológicas (Próximas 6 horas)*\n- 🌡️ Temperatura: 20.7°C\n- 💧 Humedad: 63%\n- 🌬️ Viento: 11.0 km/h\n- 🌧️ Precipitación: 3%\n\n*Última atualización: 2026-09-16 11:24:54\n\n\n### 📝 Descripción\nEl 16 de septiembre, Deke Dickerson & Los Torontos actuarán en el Kafe Antzokia de Bilbao.\n\n---\n\n\n## 🎭 Bilbo Zientzia Plaza 2026: Docufórum "Natura Bizia"\n\n---\n\n![Event image](https://opendata.euskadi.eus/contenidos/evento/2026082711114120/es_def/images/101.jpg)\n\n**📅 Fecha:** 16 de sep\n**🕒 Horario:** 18:30-20:30\n**🏙️ Municipio:** Bilbao  \n**📍 Lugar:** Biblioteca de Bilbao (Biblioteca Central de Bidebarrieta)  \n**🎟️ Tipo de evento:** Cine y audiovisuales  \n**🗺️ Mapa:** [Pincha para ver en Maps](https://www.google.com/maps?q=43.256963,-2.923441)  \n**🗣️ Idioma:** ES  \n**💶 Precio:** Gratis  \n**🌐 Web:** [Pincha para ver el enlace](nan)\n\n\n### 🌤️ Condiciones meteorológicas (Próximas 6 horas)*\n- 🌡️ Temperatura: 20.7°C\n- 💧 Humedad: 63%\n- 🌬️ Viento: 11.0 km/h\n- 🌧️ Precipitación: 3%\n\n*Última atualización: 2026-09-16 11:24:54\n\n\n### 📝 Descripción\nBilbo Zientzia Plaza 2026 arrancará el miércoles 16 de septiembre con la proyección del documental Natura Bizia (2021), escrito y dirigido por Lexeia Larrañaga y producido por Avis Productions. La sesión comenzará a las 18:30 horas en la Biblioteca de Bidebarrieta de Bilbao. La entrada será libre hasta completar aforo.\r\nCon una duración de 80 minutos, el documental propone un viaje por Euskadi y Navarra, territorios que destacan por su riqueza natural y biodiversidad. \xa0A través de la voz de José María del Río, narrador de numerosos documentales sobre naturaleza, la película invita a descubrir algunos de los espacios naturales mejor conservados de ambos territorios, así como a la extraordinaria diversidad de especies que habitan en estos ecosistemas. El documental recibió el Premio de Comunicación Ambiental en los Premios Periodismo Vasco de 2025.\r\nAcantilados verticales, un mar infinito, bosques llenos de magia y grandes formaciones de piedra caliza sirven de escenario a una obra que ofrece una mirada a la naturaleza en su estado más salvaje. En 2022 National Geographic adquirió los derechos de emisión del documental a nivel mundial y desde entonces el documental se ha emitido en todos los continentes a través de su canal Nat Geo Wild.\r\nTras la proyección tendrá lugar un coloquio de aproximadamente media hora en el que participarán sus creadores: Lexeia Larrañaga, directora y guionista del documental, y Alex Gutierréz, director de producción y cámara. Esta actividad estará moderada por la periodista Eva Caballero, directora del programa La Mecánica del Caracol de Radio Euskadi.\r\nEl docufórum forma parte del ciclo de conferencias Bidebarrieta Científica, impulsado por la Cátedra de Cultura Científica de la EHU y la Biblioteca de Bidebarrieta, entidad dependiente del área de Cultura del Ayuntamiento de Bilbao. El evento se podrá seguir tanto de manera presencial como vía streaming a través del canal Bidebarrieta Kulturgunea (YouTube). Tanto la proyección como el coloquio serán en castellano.\r\n\xa0\r\nEsta actividad se enmarca dentro del festival de divulgación científica Bilbo Zientzia Plaza 2026 (BZP). Del 16 de septiembre al 4 de octubre, la novena edición de BZP volverá a llenar Bilbao de ciencia con un amplio programa de conferencias, exposiciones, espectáculos, talleres y otras actividades.\n\n---\n\n\n## 🎭 Eszena Kalera 2026: "Libre"\n\n---\n\n![Event image](https://opendata.euskadi.eus/contenidos/evento/2026090110325360/es_def/images/32.jpg)\n\n**📅 Fecha:** 16 de sep\n**🕒 Horario:** 19:00\n**🏙️ Municipio:** Bilbao  \n**📍 Lugar:** Plaza Zumarraga (Solokoetxe)  \n**🎟️ Tipo de evento:** Teatro  \n**🗺️ Mapa:** [Pincha para ver en Maps](https://www.google.com/maps?q=43.256963,-2.923441)  \n**🗣️ Idioma:** nan  \n**💶 Precio:** Gratis  \n**🌐 Web:** [Pincha para ver el enlace](https://kulturabarrutik.eus/programa/eszena-kalera-2026/)\n\n\n### 🌤️ Condiciones meteorológicas (Próximas 6 horas)*\n- 🌡️ Temperatura: 20.7°C\n- 💧 Humedad: 63%\n- 🌬️ Viento: 11.0 km/h\n- 🌧️ Precipitación: 3%\n\n*Última atualización: 2026-09-16 11:24:54\n\n\n### 📝 Descripción\nDentro de la programación de Eszena Kalera 2026.\r\nLibre es un espectáculo de teatro visual, máscaras y títeres sobre la necesidad de encontrar un equilibrio entre los hilos que nos unen y los que nos atan a lo que queremos.\r\nLa historia del vínculo entre una mujer y un pajarito que a través de cuerdas, cestas y tramas hacen nidos y deshacen jaulas.\r\nUna pieza que invita a entrelazarnos para tejer un refugio donde cada cual sea quien quiera ser.\r\n\r\n\r\nFicha artística:\xa0\r\n\r\n\r\n\r\n\r\nCreación e interpretación:\xa0María Arriera\r\nMirada externa: Amaia Garrosa\r\nEspacio sonoro: Samuel Cano Braojos\r\nEscenografía, máscaras y títeres: Compañía Arriera\r\nProducción: Compañía Arriera\n\n---\n\n\n## 🎭 Presentación de libro: "Yo quiero no morir"\n\n---\n\n![Event image](https://opendata.euskadi.eus/contenidos/evento/2026090313394438/es_def/images/34.jpg)\n\n**📅 Fecha:** 16 de sep\n**🕒 Horario:** 19:00\n**🏙️ Municipio:** Bilbao  \n**📍 Lugar:** Librería Joker  \n**🎟️ Tipo de evento:** Conferencia  \n**🗺️ Mapa:** [Pincha para ver en Maps](https://www.google.com/maps?q=43.256963,-2.923441)  \n**🗣️ Idioma:** ES  \n**💶 Precio:** 3 €  \n**🌐 Web:** [Pincha para ver el enlace](https://www.jokercomics.es/agenda/evento.php?codigo=471)\n\n\n### 🌤️ Condiciones meteorológicas (Próximas 6 horas)*\n- 🌡️ Temperatura: 20.7°C\n- 💧 Humedad: 63%\n- 🌬️ Viento: 11.0 km/h\n- 🌧️ Precipitación: 3%\n\n*Última atualización: 2026-09-16 11:24:54\n\n\n### 📝 Descripción\nPresentación del libro "Yo no quiero morir".\r\nEl importe de la reserva se descontará en la próxima compra en la librería.\n\n---\n\n\n## 🎭 Presentación de libro: "HOMO MEDIATICUS" (CARLOS A. SCOLARI)\n\n---\n\n![Event image](https://opendata.euskadi.eus/contenidos/evento/2026091412475444/es_def/images/1.jpg)\n\n**📅 Fecha:** 16 de sep\n**🕒 Horario:** 19:00\n**🏙️ Municipio:** Bilbao  \n**📍 Lugar:** La Ilusa  \n**🎟️ Tipo de evento:** Conferencia  \n**🗺️ Mapa:** [Pincha para ver en Maps](https://www.google.com/maps?q=43.256963,-2.923441)  \n**🗣️ Idioma:** ES  \n**💶 Precio:** Gratis  \n**🌐 Web:** [Pincha para ver el enlace](nan)\n\n\n### 🌤️ Condiciones meteorológicas (Próximas 6 horas)*\n- 🌡️ Temperatura: 20.7°C\n- 💧 Humedad: 63%\n- 🌬️ Viento: 11.0 km/h\n- 🌧️ Precipitación: 3%\n\n*Última atualización: 2026-09-16 11:24:54\n\n\n### 📝 Descripción\nEl miércoles 16 de septiembre a las 19:00h presentación del libro "Homo Mediaticus. Una historia de la humanidad: del "hashtag" neandertal al Ipod" en la librería La ilusa. El autor, Carlos A. Scolari, estará acompañado de Simón Peña Fernández (UPV/EHU).\r\nDirección: C/Hernani, 8. Bilbao\n\n---\n\n\n## 🎭 Presentación de libro: "Zaldibarko berbeta eta literatura: Zalduko urak"\n\n---\n\n![Event image](https://opendata.euskadi.eus/contenidos/evento/2026091511465858/es_def/images/24.jpg)\n\n**📅 Fecha:** 16 de sep\n**🕒 Horario:** 11:00\n**🏙️ Municipio:** Bilbao  \n**📍 Lugar:** Euskaltzaindiaren Aretoa  \n**🎟️ Tipo de evento:** Conferencia  \n**🗺️ Mapa:** [Pincha para ver en Maps](https://www.google.com/maps?q=43.256963,-2.923441)  \n**🗣️ Idioma:** EU  \n**💶 Precio:** Gratis  \n**🌐 Web:** [Pincha para ver el enlace](nan)\n\n\n### 🌤️ Condiciones meteorológicas (Próximas 6 horas)*\n- 🌡️ Temperatura: 20.7°C\n- 💧 Humedad: 63%\n- 🌬️ Viento: 11.0 km/h\n- 🌧️ Precipitación: 3%\n\n*Última atualización: 2026-09-16 11:24:54\n\n\n### 📝 Descripción\nJoseba Sarrionandia eta Gaizka Zabarte idazleen Zaldibarko berbeta eta literatura: Zalduko urak liburua aurkeztuko da.\r\nSinopsia:\r\nEstatua bere indar guztiekin sartu zen Zaldibarrera ere, tsunami bat bezala, XIX. mendean batez ere. Olatu erraldoi hori areagotu egin zen, modu zitalean, XX. mendean. Eta oraindik ere hementxe dabil uholdea, bertako hizkuntza era batean edo bestean desplazatzen, ordezkatzen eta itotzen.\r\nLiburu hau Zaldibarko euskararen eta biztanleen memoria errekuperatzeko ahalegina da. Oinetako zolekin ibileran ukitzen diren leku-izenak, kanta zahar ja erdi ahaztuak, Mari Urrikeren eta beste arima beldurgarrien ipuinak, gerraurreko prentsako kronika abertzaleak, euskaraz eskapatu den dokumentu ofizialen bat, umeen- eta gerra testigantzak, gerraondoko bizimodu gogorrarenak, idazle ezezagun eta ezagun batzuen kontuak, Jose Luis Mendilibar Greziako zaldibartarrarenak adibidez. Testu zaharrak eta berriak, gaur eguneko eskolako umeak arte...\r\nMundu guztia erdi-birtualki eta airean bezala bizitzen hasi denean, gure erreferentzia kulturalak gero eta lausoagoak direnean, ez da kaltegarria inguru hurbilari begiratzea. Lokala ala globala dikotomia faltsua da, ondo lokalak izanez gero gara benetan unibertsalak, zapaltzen dugun lurra eta inguruan hodeiertzeraino dugun paisaia ezaguturik.\n\n---\n'

In [89]:
try:
    results=list(Filter.T.to_dict().values())
    
except:
    #results=[]
    results=list(I.T.to_dict().values())
try:
    if len(results)!=0:
        print(f"""---
Abajo te enviamos los resultados de tu búsqueda.

{format_events_md(results)}
---""")
    else:
        results=list(I.T.to_dict().values())
        print(print(f"""---
⚠️ No hemos encontrado información que coincida con los filtros de tu búsqueda.

Te recomendamos reformular la consulta o intentar describirla de otra manera.

{format_events_md(results)}
---"""))

except Exception as e:
    error_msg = (
        """---
⚠️ No hemos encontrado información que coincida con los filtros de tu búsqueda.

Te recomendamos reformular la consulta o intentar describirla de otra manera.

---"""
    )
    print(error_msg)


---
⚠️ No hemos encontrado información que coincida con los filtros de tu búsqueda.

Te recomendamos reformular la consulta o intentar describirla de otra manera.

---


In [80]:
type(test)

NoneType

In [58]:
make_assignment(ask_model("eventos en bilbao"))

"I[I.municipalityEs == 'Bilbao']"

In [39]:
df["priceEs"].unique()

<ArrowStringArray>
[                     'Gratis',                           nan,
                   '20 / 41 €',                     'Reserva',
                   '20 / 24 €',        'Gratis (inscripción)',
                         '3 €',     'Gratis (con invitación)',
              'Con invitación',    'Gratis (con inscripción)',
 'Gratis (previa inscripción)',                         '10€']
Length: 12, dtype: str

In [31]:
pd.DataFrame()

""
